# Brain Age Prediction Training Notebook per Kaggle
Questo notebook segue le procedure rigorose descritte nel paper **SFCN** (Peng et al., 2021):
- Ottimizzatore SGD (LR=0.01) con decadimento dello 0.3 ogni 30 epoche.
- KL Divergence Loss (Soft-classification) convertendo la tua classe categorica in una distribuzione d'età continua.
- Data Augmentation: 50% shift random (-2, +2) in tutti gli assi e 50% probabilità di flip sagittale.
- Modello selezionato in base al miglior **MAE**.

In [ ]:
!rm -rf SFCN
!git clone https://github.com/PietroSchgor/SFCN.git

import sys
sys.path.append('/kaggle/working/SFCN')

In [ ]:
import os
import json
import glob
import numpy as np
import nibabel as nib
import torch
from torch.utils.data import Dataset, DataLoader

# Import dal tuo repository
from dp_model.model_files.sfcn import SFCN
from dp_model import dp_utils as dpu
from train import train_model

## 1. Dataset Custom (con Data Augmentation)

In [ ]:
class BrainAgeDataset(Dataset):
    def __init__(self, data_dir, is_train=True):
        self.data_dir = data_dir
        self.is_train = is_train
        self.subject_dirs = sorted(glob.glob(os.path.join(data_dir, "sub-*")))
        self.samples = []
        
        # Range per la soft-label: da 0 a 70 anni (per coprire le tue classi 1-65)
        self.bin_range = [0, 70]
        self.bin_step = 1
        self.sigma = 1.0
        
        for subj_dir in self.subject_dirs:
            subj_id = os.path.basename(subj_dir)
            nii_path = os.path.join(subj_dir, f"{subj_id}_FLAIR_MNI152_1mm.nii")
            
            if not os.path.exists(nii_path):
                nii_path = nii_path + ".gz"
                if not os.path.exists(nii_path):
                    print(f"Saltato {subj_id}: NIfTI non trovato in {nii_path}")
                    continue
                    
            json_path = os.path.join(subj_dir, f"{subj_id}_participant_info.json")
            if not os.path.exists(json_path):
                print(f"Saltato {subj_id}: JSON non trovato in {json_path}")
                continue
                
            with open(json_path, 'r') as f:
                info = json.load(f)
                
            participant_info = info.get("participant_info", {})
            age_cat_val = participant_info.get("age_scan")
            
            if age_cat_val is None:
                print(f"Saltato {subj_id}: Chiave 'age_scan' assente in 'participant_info'.")
                continue
            
            try:
                # Mappa la tua classe da 1-13 ad un'età continua, calcolando il valore medio della fascia
                # Es: classe 1 (1-5 anni) -> media 3 anni, classe 2 (6-10) -> media 8
                age_cat = int(age_cat_val) - 1
                true_age = 3 + age_cat * 5
                
                # Trasforma l'età esatta in una probabilità Gaussiana (vettore di 70 elementi)
                y, _ = dpu.num2vect(true_age, self.bin_range, self.bin_step, self.sigma)
                
            except Exception as e:
                print(f"Saltato {subj_id}: Errore parsing age_scan = {age_cat_val}. Errore: {e}")
                continue
            
            self.samples.append({
                "nii_path": nii_path,
                "label_vect": y,
                "true_age": true_age
            })

        if len(self.samples) == 0:
            print(f"ATTENZIONE: Nessun campione trovato. Cerca le cartelle sub-* in: {data_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        img = nib.load(sample['nii_path'])
        data = img.get_fdata(dtype=np.float32)
        
        mean_val = np.mean(data)
        if mean_val > 0:
            data = data / mean_val
            
        in_sp = data.shape
        out_sp = (160, 192, 160)
        
        if self.is_train:
            # 1. 50% probabilità di flip sagittale (asse 0 nel NIfTI)
            if np.random.rand() > 0.5:
                data = np.flip(data, axis=0).copy()
                
            # 2. Random shift -2, -1, 0, 1, 2 voxel per tutti gli assi
            dx = np.random.randint(-2, 3)
            dy = np.random.randint(-2, 3)
            dz = np.random.randint(-2, 3)
        else:
            dx, dy, dz = 0, 0, 0
            
        # Eseguiamo il cropping al centro APPLICANDO il random shift
        x_c = int((in_sp[0] - out_sp[0]) / 2) + dx
        y_c = int((in_sp[1] - out_sp[1]) / 2) + dy
        z_c = int((in_sp[2] - out_sp[2]) / 2) + dz
        
        data = data[x_c:x_c+out_sp[0], y_c:y_c+out_sp[1], z_c:z_c+out_sp[2]]
        data = np.expand_dims(data, axis=0)
        
        tensor_data = torch.from_numpy(data)
        label_vect = torch.tensor(sample['label_vect'], dtype=torch.float32)
        
        return tensor_data, label_vect, sample['true_age']

## 2. Avvio del Training (Paper SFCN parameters)

In [ ]:
KAGGLE_DATA_DIR = "/kaggle/input/datasets/elenaschgor/dataset-2-t1-flair/ds004199_final/"

# Creiamo Datasets separati per train (con augmentation) e val (senza augmentation)
full_train_dataset = BrainAgeDataset(KAGGLE_DATA_DIR, is_train=True)
full_val_dataset   = BrainAgeDataset(KAGGLE_DATA_DIR, is_train=False)
print(f"Trovati {len(full_train_dataset)} campioni validi.\n")

if len(full_train_dataset) > 0:
    train_size = int(0.8 * len(full_train_dataset))
    val_size = len(full_train_dataset) - train_size
    
    # Ci assicuriamo che gli indici coincidano per train e val
    indices = torch.randperm(len(full_train_dataset)).tolist()
    
    train_dataset = torch.utils.data.Subset(full_train_dataset, indices[:train_size])
    val_dataset = torch.utils.data.Subset(full_val_dataset, indices[train_size:])
    
    # Batch size 8 (o adattato per VRAM limitata se 8 OOM, es. scendi a 4)
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Utilizzo dispositivo: {device}\n")
    
    # Inizializza il modello. Siccome abbiamo usato bins [0, 70], l'output_dim deve essere 70
    model = SFCN(output_dim=70).to(device)
    
    # Ottimizzatore SGD (SFCN Paper)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, weight_decay=0.001)
    
    # Lancia l'addestramento importato da train.py
    # (In train_model è già presente lo StepLR e la best MAE selection)
    trained_model, train_losses, val_losses, val_maes = train_model(
        model=model, 
        train_loader=train_loader, 
        val_loader=val_loader, 
        optimizer=optimizer, 
        device=device, 
        epochs=130
    )
